In [0]:
dbutils.widgets.removeAll()

In [0]:
import json
from pyspark.sql.functions import explode, col
import pydantic

In [0]:
%run "./MoveFileToConsolidation"

In [0]:
dbutils.widgets.text("ConsolidationJSON", "")
ConsolidationJSON = dbutils.widgets.get("ConsolidationJSON")

print(f"Received ConsolidationJSON parameter (length: {len(ConsolidationJSON)})")

In [0]:
# Load and explode input JSON parameters (serverless-compatible)
print("Step 1: Parsing JSON and creating DataFrame...")
if not ConsolidationJSON or ConsolidationJSON.strip() == "":
    raise ValueError("ConsolidationJSON widget is empty. Please provide a valid JSON string.")

try:
    parsed_json = json.loads(ConsolidationJSON)
except json.JSONDecodeError as e:
    raise ValueError(f"Invalid JSON format provided in widget: {e}")

file_id_df = spark.createDataFrame([parsed_json])

print("Step 2: Exploding FileIds array...")
exploded_file_ids = file_id_df.select(explode(col("FileIds")).alias("col")).select(
    col("col.DataGroupTrackingID"),
    col("col.DataGroupMappingId"),
    col("col.FileId"),
    col("col.FileLayoutID"),
    col("col.FileLayoutDescription"),
    col("col.CurrentContainer"),
    col("col.CurrentFolderPath"),
    col("col.ConsolidatedMappingFilePath"),
    col("col.ConsolidatedMappingFileName"),
    col("col.ConsolidatedLayerDataModelFilePath"),
    col("col.ConsolidatedLayerDataModel"),
    col("col.ConsolidatedFolderPath")
)

file_count = exploded_file_ids.count()
print(f"Found {file_count} file(s) to process")

In [0]:
rJSON_list = []

# Collect data rows to loop securely using native DataFrame actions
print("\nStep 3: Collecting rows for processing...")
rows = exploded_file_ids.collect()
print(f"Collected {len(rows)} row(s)")

In [0]:
print(f"\nStep 4: Processing each file...")
for idx, t in enumerate(rows, 1):
    print(f"\n--- Processing file {idx}/{len(rows)}: FileId={t['FileId']} ---")
    results = ""
    try:
        # Call process_consolidation function directly (no separate execution, no concurrency limit)
        print(f"  Calling process_consolidation function...")
        results = process_consolidation(
            FileId=str(t["FileId"]),
            CurrentContainer=str(t["CurrentContainer"]),
            CurrentFolderPath=str(t["CurrentFolderPath"]),
            ConsolidatedLayerDataModel=str(t["ConsolidatedLayerDataModel"]),
            ConsolidatedLayerDataModelFilePath=str(t["ConsolidatedLayerDataModelFilePath"]),
            ConsolidatedMappingFileName=str(t["ConsolidatedMappingFileName"]),
            ConsolidatedMappingFilePath=str(t["ConsolidatedMappingFilePath"]),
            ConsolidatedFolderPath=str(t["ConsolidatedFolderPath"])
        )
        print(f"  process_consolidation completed")
        print(f"  Raw result (first 200 chars): {results[:10]}...")
    except Exception as e:
        print(f"  process_consolidation failed: {str(e)}")
        results = json.dumps({
            "CurrentJobId": "None",
            "ConsolidatedCount": "0",
            "Status": "FAILED",
            "ErrorMessage": str(e)
        })

    # Process output metrics
    try:
        res_data = json.loads(results)
        print(f"  Result: {res_data.get('Status', 'UNKNOWN')} - {res_data.get('ConsolidatedCount', '0')} rows")
    except Exception as parse_err:
        res_data = {"CurrentJobId": "Error", "ConsolidatedCount": "0", "Status": "FAILED", "ErrorMessage": f"Failed to parse JSON: {str(parse_err)}. Raw result: {results[:10]}"}
        print(f"  Failed to parse result JSON: {str(parse_err)}")

    record = {
        "FileID": t["FileId"],
        "DataGroupTrackingID": t["DataGroupTrackingID"],
        "DataGroupMappingId": t["DataGroupMappingId"],
        "CurrentContainer": t["CurrentContainer"],
        "CurrentFolderPath": t["CurrentFolderPath"],
        "ConsolidatedFolderPath": t["ConsolidatedFolderPath"],
        "CurrentJobId": res_data.get("CurrentJobId", ""),
        "ConsolidatedCount": str(res_data.get("ConsolidatedCount", "0")),
        "Status": res_data.get("Status", ""),
        "ErrorMessage": res_data.get("ErrorMessage", "")
    }
    rJSON_list.append(record)

    print(f"\n All files processed. Total: {len(rJSON_list)}")

# Return the aggregate JSON string output back
result_json = json.dumps(rJSON_list)
print(f"\nReturning result JSON (length: {len(result_json)} chars)")
dbutils.notebook.exit(result_json)